In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
from itertools import product
import calendar

In [2]:
BASE_DIR = "dataset"

dfs = {}

for file in os.listdir(BASE_DIR):
    if file.endswith(".csv"):
        path = os.path.join(BASE_DIR, file)

        df = pd.read_csv(path)

        name = file.replace(".csv", "")
        dfs[name] = df
dfs.keys()
tables_name = dfs.keys()-{'data_dictionary'}


# EDA 1

In [3]:
table_info = []
for name in tables_name:
    data = dfs[name]
    table_info.append(
        {"Table":name,
         "Rows":data.shape[0],
         "Columns":data.shape[1],
         "missing_value":data.isna().sum().sum(),
         "duplicated data":data.duplicated().sum()
         }
    )
display(pd.DataFrame(table_info))
    

,Table,Rows,Columns,missing_value,duplicated data
0,sales_teams,35,3,0,0
1,products,7,3,0,0
2,sales_pipeline,8800,8,5015,0
3,accounts,85,7,70,0


In [4]:
table_columns_type = []
for name in tables_name:
    data = dfs[name].dtypes
    for column,type in data.items():
        table_columns_type.append(
            {"table":name,
            "column":column,
            "type of data":type,
            "missing values": dfs[name][column].isna().sum(),
            "unique values": dfs[name][column].nunique()}
        )
data_table_info = pd.DataFrame(table_columns_type)
data_table_info

,table,column,type of data,missing values,unique values
0,sales_teams,sales_agent,str,0,35
1,sales_teams,manager,str,0,6
2,sales_teams,regional_office,str,0,3
3,products,product,str,0,7
4,products,series,str,0,3
5,products,sales_price,int64,0,7
6,sales_pipeline,opportunity_id,str,0,8800
7,sales_pipeline,sales_agent,str,0,30
8,sales_pipeline,product,str,0,7
9,sales_pipeline,account,str,337,85


# Data Cleaning

## 
Accounts Table

In [5]:
df_accounts = dfs['accounts']
display(data_table_info[data_table_info['table']=="accounts"])
df_accounts.loc[df_accounts['subsidiary_of'].isna(),'subsidiary_of'] = 'Unknown'
columns = (df_accounts.select_dtypes(include=['object','str'])).columns
for column in columns:
    df_accounts[column] = df_accounts[column].str.strip()
display(df_accounts)
display(df_accounts.info())
display(df_accounts.describe())

,table,column,type of data,missing values,unique values
14,accounts,account,str,0,85
15,accounts,sector,str,0,10
16,accounts,year_established,int64,0,35
17,accounts,revenue,float64,0,85
18,accounts,employees,int64,0,85
19,accounts,office_location,str,0,15
20,accounts,subsidiary_of,str,70,7


,account,sector,year_established,revenue,employees,office_location,subsidiary_of
0,Acme Corporation,technolgy,1996,1100.04,2822,United States,Unknown
1,Betasoloin,medical,1999,251.41,495,United States,Unknown
2,Betatech,medical,1986,647.18,1185,Kenya,Unknown
3,Bioholding,medical,2012,587.34,1356,Philipines,Unknown
4,Bioplex,medical,1991,326.82,1016,United States,Unknown
...,...,...,...,...,...,...,...
80,Zathunicon,retail,2010,71.12,144,United States,Unknown
81,Zencorporation,technolgy,2011,40.79,142,China,Unknown
82,Zoomit,entertainment,1992,324.19,978,United States,Unknown
83,Zotware,software,1979,4478.47,13809,United States,Unknown


<class 'pandas.DataFrame'>
RangeIndex: 85 entries, 0 to 84
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   account           85 non-null     str    
 1   sector            85 non-null     str    
 2   year_established  85 non-null     int64  
 3   revenue           85 non-null     float64
 4   employees         85 non-null     int64  
 5   office_location   85 non-null     str    
 6   subsidiary_of     85 non-null     str    
dtypes: float64(1), int64(2), str(4)
memory usage: 4.8 KB


None

,year_established,revenue,employees
count,85.000000,85.000000,85.000000
mean,1996.105882,1994.632941,4660.823529
std,8.865427,2169.491436,5715.601198
min,1979.000000,4.540000,9.000000
25%,1989.000000,497.110000,1179.000000
50%,1996.000000,1223.720000,2769.000000
75%,2002.000000,2741.370000,5595.000000
max,2017.000000,11698.030000,34288.000000


##
Products Table

In [6]:
df_products = dfs['products']
display(data_table_info[data_table_info['table']=='products'])
columns = (df_products.select_dtypes(include = 'str')).columns
for column in columns:
    df_products[column] = df_products[column].str.strip()
display(df_products)
display(df_products.describe())

,table,column,type of data,missing values,unique values
3,products,product,str,0,7
4,products,series,str,0,3
5,products,sales_price,int64,0,7


,product,series,sales_price
0,GTX Basic,GTX,550
1,GTX Pro,GTX,4821
2,MG Special,MG,55
3,MG Advanced,MG,3393
4,GTX Plus Pro,GTX,5482
5,GTX Plus Basic,GTX,1096
6,GTK 500,GTK,26768


,sales_price
count,7.000000
mean,6023.571429
std,9388.428070
min,55.000000
25%,823.000000
50%,3393.000000
75%,5151.500000
max,26768.000000


##
Sales Pipeline Table

In [7]:
df_sales_pipeline = dfs['sales_pipeline']
display(data_table_info[data_table_info['table']=='sales_pipeline'])
column_str = df_sales_pipeline.select_dtypes(
    include=['object', 'str']
).columns

for column in column_str:
    df_sales_pipeline[column] = df_sales_pipeline[column].str.strip()

display(pd.DataFrame(df_sales_pipeline['deal_stage'].value_counts()))

display(pd.crosstab(
    df_sales_pipeline['deal_stage'],
    df_sales_pipeline['account'].isna()
))

display(pd.crosstab(
    df_sales_pipeline['deal_stage'],
    df_sales_pipeline['engage_date'].isna()
))
display(pd.crosstab(
    df_sales_pipeline['deal_stage'],
    df_sales_pipeline['close_date'].isna()
))

,table,column,type of data,missing values,unique values
6,sales_pipeline,opportunity_id,str,0,8800
7,sales_pipeline,sales_agent,str,0,30
8,sales_pipeline,product,str,0,7
9,sales_pipeline,account,str,337,85
10,sales_pipeline,deal_stage,str,0,4
11,sales_pipeline,engage_date,str,500,421
12,sales_pipeline,close_date,str,2089,306
13,sales_pipeline,close_value,float64,2089,2051


,count
deal_stage,
Won,4238
Lost,2473
Engaging,1589
Prospecting,500


account,False,True
deal_stage,,
Engaging,1589,0
Lost,2473,0
Prospecting,163,337
Won,4238,0


engage_date,False,True
deal_stage,,
Engaging,1589,0
Lost,2473,0
Prospecting,0,500
Won,4238,0


close_date,False,True
deal_stage,,
Engaging,0,1589
Lost,2473,0
Prospecting,0,500
Won,4238,0


Given that the occurrence of missing values in the sales pipeline dataset is associated with specific pipeline stages, the dataset will be segmented by deal stage to facilitate a more systematic assessment and determine the most appropriate missing value treatment for each segment.

###
Prospecting Sales Pipeline Table

In [8]:
df_sales_pipeline_Pros = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Prospecting'].reset_index(drop=True) 

df_sales_pipeline_Pros.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  500 non-null    str    
 1   sales_agent     500 non-null    str    
 2   product         500 non-null    str    
 3   account         163 non-null    str    
 4   deal_stage      500 non-null    str    
 5   engage_date     0 non-null      str    
 6   close_date      0 non-null      str    
 7   close_value     0 non-null      float64
dtypes: float64(1), str(7)
memory usage: 31.4 KB


At the Prospecting stage, some records contain account information, while others have missing values. This variation is considered consistent with the nature of prospecting, where potential customers may not yet be fully identified or linked to an existing account.

The missing values are therefore not treated as invalid records but as potentially meaningful indicators of the prospect identification process.

- **Available account value:** The potential customer has been identified and linked to an account.
- **Missing account value:** The potential customer may not yet have been identified or linked to an account.

Based on this interpretation, records with missing account values will be retained to avoid unnecessary data loss. Instead, missing values will be replaced with the category **"Unidentified"** to distinguish them from identified accounts while preserving the records for subsequent analysis.

Missing values in the `engage_date`, `close_date`, and `close_value` columns of `df_sales_pipeline_Pros` were not treated, as opportunities in the Prospecting stage are not expected to have engagement dates, closing dates, or closing values yet.

In [9]:
df_sales_pipeline_Pros.loc[df_sales_pipeline_Pros['account'].isna(),'account']= 'Unidentified'
df_sales_pipeline_Pros.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  500 non-null    str    
 1   sales_agent     500 non-null    str    
 2   product         500 non-null    str    
 3   account         500 non-null    str    
 4   deal_stage      500 non-null    str    
 5   engage_date     0 non-null      str    
 6   close_date      0 non-null      str    
 7   close_value     0 non-null      float64
dtypes: float64(1), str(7)
memory usage: 31.4 KB


###
Engaging Sales Pipeline Table

In [10]:
df_sales_pipeline_engaging = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Engaging'].reset_index(drop=True)
display(df_sales_pipeline_engaging.info())
display(df_sales_pipeline_engaging.head())


<class 'pandas.DataFrame'>
RangeIndex: 1589 entries, 0 to 1588
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  1589 non-null   str    
 1   sales_agent     1589 non-null   str    
 2   product         1589 non-null   str    
 3   account         1589 non-null   str    
 4   deal_stage      1589 non-null   str    
 5   engage_date     1589 non-null   str    
 6   close_date      0 non-null      str    
 7   close_value     0 non-null      float64
dtypes: float64(1), str(7)
memory usage: 99.4 KB


None

,opportunity_id,sales_agent,product,account,deal_stage,engage_date,close_date,close_value
0,HAXMC4IX,James Ascencio,MG Advanced,Inity,Engaging,2016-11-03,NaN,NaN
1,UP409DSB,Maureen Marcano,MG Advanced,Ganjaflex,Engaging,2016-11-10,NaN,NaN
2,EG7OFLFR,Kami Bicknell,GTX Basic,Zumgoity,Engaging,2016-11-14,NaN,NaN
3,OLVI7L8M,Cassey Cress,GTXPro,Donware,Engaging,2016-11-16,NaN,NaN
4,F5U1ACDD,Kami Bicknell,GTX Plus Basic,Lexiqvolax,Engaging,2016-11-19,NaN,NaN


Won Sales Pipeline Table

In [11]:
df_sales_pipeline_Won = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Won'].reset_index(drop=True)
date_column = ['close_date','engage_date']
for column in date_column:
    df_sales_pipeline_Won[column] = pd.to_datetime(df_sales_pipeline_Won[column])

display(df_sales_pipeline_Won.info())
display(df_sales_pipeline_Won.describe())


<class 'pandas.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  4238 non-null   str           
 1   sales_agent     4238 non-null   str           
 2   product         4238 non-null   str           
 3   account         4238 non-null   str           
 4   deal_stage      4238 non-null   str           
 5   engage_date     4238 non-null   datetime64[us]
 6   close_date      4238 non-null   datetime64[us]
 7   close_value     4238 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 265.0 KB


None

,engage_date,close_date,close_value
count,4238,4238,4238.000000
mean,2017-06-09 12:50:58.046248,2017-07-31 07:37:00.481359,2360.909391
min,2016-10-20 00:00:00,2017-03-01 00:00:00,38.000000
25%,2017-03-25 00:00:00,2017-05-18 00:00:00,518.000000
50%,2017-06-12 00:00:00,2017-08-02 00:00:00,1117.000000
75%,2017-09-01 00:00:00,2017-10-16 00:00:00,4429.500000
max,2017-12-27 00:00:00,2017-12-31 00:00:00,30288.000000
std,NaN,NaN,2544.484463


In [12]:
df_sales_pipeline_Lost = df_sales_pipeline[df_sales_pipeline['deal_stage']=='Lost'].reset_index(drop=True)

date_column= ['engage_date','close_date']
for column in date_column:
    df_sales_pipeline_Lost[column] = pd.to_datetime(df_sales_pipeline_Lost[column])

df_sales_pipeline_Lost = pd.merge(df_sales_pipeline_Lost,df_products[['product','sales_price']],on ='product',how='left')
df_sales_pipeline_Lost.rename(
    columns={'sales_price': 'close_values_lost'},
    inplace=True
)

display(df_sales_pipeline_Lost.info())
display(df_sales_pipeline_Lost.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2473 entries, 0 to 2472
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   opportunity_id     2473 non-null   str           
 1   sales_agent        2473 non-null   str           
 2   product            2473 non-null   str           
 3   account            2473 non-null   str           
 4   deal_stage         2473 non-null   str           
 5   engage_date        2473 non-null   datetime64[us]
 6   close_date         2473 non-null   datetime64[us]
 7   close_value        2473 non-null   float64       
 8   close_values_lost  2055 non-null   float64       
dtypes: datetime64[us](2), float64(2), str(5)
memory usage: 174.0 KB


None

,engage_date,close_date,close_value,close_values_lost
count,2473,2473,2473.0,2055.000000
mean,2017-06-22 02:11:00.897695,2017-08-02 13:41:01.625556,0.0,1913.036496
min,2016-11-04 00:00:00,2017-03-01 00:00:00,0.0,55.000000
25%,2017-04-12 00:00:00,2017-05-19 00:00:00,0.0,550.000000
50%,2017-06-29 00:00:00,2017-08-02 00:00:00,0.0,1096.000000
75%,2017-09-08 00:00:00,2017-10-19 00:00:00,0.0,3393.000000
max,2017-12-22 00:00:00,2017-12-31 00:00:00,0.0,26768.000000
std,NaN,NaN,0.0,2537.528974


In [13]:
df_sales_pipeline_eng = df_sales_pipeline[~df_sales_pipeline['engage_date'].isna()]
for column in date_column:
    df_sales_pipeline_eng[column] = pd.to_datetime(df_sales_pipeline_eng[column])
df_sales_pipeline_eng.info()


<class 'pandas.DataFrame'>
RangeIndex: 8300 entries, 0 to 8299
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  8300 non-null   str           
 1   sales_agent     8300 non-null   str           
 2   product         8300 non-null   str           
 3   account         8300 non-null   str           
 4   deal_stage      8300 non-null   str           
 5   engage_date     8300 non-null   datetime64[us]
 6   close_date      6711 non-null   datetime64[us]
 7   close_value     6711 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 518.9 KB


In [14]:
df_sales_pipeline_closed = df_sales_pipeline[~df_sales_pipeline['close_date'].isna()]
for column in date_column:
    df_sales_pipeline_closed[column] = pd.to_datetime(df_sales_pipeline_closed[column])
df_sales_pipeline_closed.info()


<class 'pandas.DataFrame'>
Index: 6711 entries, 0 to 8299
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   opportunity_id  6711 non-null   str           
 1   sales_agent     6711 non-null   str           
 2   product         6711 non-null   str           
 3   account         6711 non-null   str           
 4   deal_stage      6711 non-null   str           
 5   engage_date     6711 non-null   datetime64[us]
 6   close_date      6711 non-null   datetime64[us]
 7   close_value     6711 non-null   float64       
dtypes: datetime64[us](2), float64(1), str(5)
memory usage: 471.9 KB


###
Sales Teams Table

In [15]:
df_sales_teams = dfs['sales_teams']
display(data_table_info[data_table_info['table']=="sales_teams"])
columns = df_sales_teams.columns
for column in columns:
    df_sales_teams[column] = df_sales_teams[column].str.strip()
display(df_sales_teams.head())
display(df_sales_teams.info())
display(df_sales_teams.describe())

,table,column,type of data,missing values,unique values
0,sales_teams,sales_agent,str,0,35
1,sales_teams,manager,str,0,6
2,sales_teams,regional_office,str,0,3


,sales_agent,manager,regional_office
0,Anna Snelling,Dustin Brinkmann,Central
1,Cecily Lampkin,Dustin Brinkmann,Central
2,Versie Hillebrand,Dustin Brinkmann,Central
3,Lajuana Vencill,Dustin Brinkmann,Central
4,Moses Frase,Dustin Brinkmann,Central


<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sales_agent      35 non-null     str  
 1   manager          35 non-null     str  
 2   regional_office  35 non-null     str  
dtypes: str(3)
memory usage: 972.0 bytes


None

,sales_agent,manager,regional_office
count,35,35,35
unique,35,6,3
top,Anna Snelling,Melvin Marxen,East
freq,1,6,12


In [17]:
clean_table = [
    df_accounts,
    df_products,
    df_sales_pipeline,
    df_sales_pipeline_Pros,
    df_sales_pipeline_engaging,
    df_sales_pipeline_Won,
    df_sales_pipeline_Lost,
    df_sales_pipeline_eng,
    df_sales_pipeline_closed,
    df_sales_teams
]

table_names = [
    'accounts',
    'products',
    'sales_pipeline',
    'sales_pipeline_Pros',
    'sales_pipeline_engaging',
    'sales_pipeline_Won',
    'sales_pipeline_Lost',
    'sales_pipeline_all_eng',
    'sales_pipeline_closed',
    'sales_teams'
]

folder = r'E:\5_Portofolio Data Analyst\5_CRM_Analysis\clean_dataset'

for name, df in zip(table_names, clean_table):
    df.to_csv(os.path.join(folder, f'{name}.csv'), index=False)